Вы можете заказать версии печати и электронных книг * Think Python 3e * из
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) и
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

# Анализ текста и поколение

На этом этапе мы рассмотрели основные структуры данных Python - списки, словари и кортежи - и некоторые алгоритмы, которые их используют.
В этой главе мы будем использовать их для изучения текстового анализа и генерации Маркова:

* Анализ текста - это способ описать статистические отношения между словами в документе, например, вероятность того, что одно слово следует другому, и

* Генерация Маркова - это способ генерировать новый текст со словами и фразами, похожими на исходный текст.

Эти алгоритмы аналогичны частям большой языковой модели (LLM), которая является ключевым компонентом чат -бота.

Мы начнем с подсчета количества раз, когда каждое слово появляется в книге.
Затем мы посмотрим на пары слов и составим список слов, которые могут следовать каждому слову.
Мы сделаем простую версию генератора Маркова, и в качестве упражнения у вас будет возможность сделать более общую версию.

## уникальные слова

В качестве первого шага к анализу текста, давайте прочитаем книгу - * Странный случай доктора Джекилла и мистера Хайда * Роберта Луи Стивенсона - и подсчитывать количество уникальных слов.
Инструкции по загрузке книги находятся в записной книжке для этой главы.

Следующая ячейка загружает книгу из Project Gutenberg.

In [3]:
download('https://www.gutenberg.org/cache/epub/43/pg43.txt');

Версия, доступная в Project Gutenberg, включает информацию о книге в начале и информацию о лицензии в конце.
Мы используем `clean_file` из главы 8, чтобы удалить этот материал и написать« чистый »файл, который содержит только текст книги.

In [4]:
def is_special_line(line):
    return line.strip().startswith('*** ')

In [5]:
def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)
        
    reader.close()
    writer.close()

In [6]:
filename = 'dr_jekyll.txt'

In [7]:
clean_file('pg43.txt', filename)

Мы используем цикл «for» для чтения строк из файла и «разделить», чтобы разделить строки на слова.
Затем, чтобы отслеживать уникальные слова, мы сохраним каждое слово в качестве ключа в словаре.

In [9]:
unique_words = {}
for line in open(filename):
    seq = line.split()
    for word in seq:
        unique_words[word] = 1

len(unique_words)

Длина словаря - это количество уникальных слов - около `6000` по этому способу подсчета.
Но если мы осмотрим их, мы увидим, что некоторые не являются действительными словами.

Например, давайте посмотрим на самые длинные слова в `in unique_words`.
Мы можем использовать «отсортированный» для сортировки слов, передавая функцию `len` в качестве аргумента ключевого слова, чтобы слова отсортированы по длине.

In [88]:
sorted(unique_words, key=len)[-5:]

Индекс среза, `[-5:]`, выбирает последние элементы сортированного списка, которые являются самыми длинными словами. 

Список включает в себя некоторые законно длинные слова, такие как «Ограничение», и некоторые дефис-слова, такие как «шоколадный цвет».
Но некоторые из самых длинных «слов» на самом деле являются двумя словами, разделенными чертой.
И другие слова включают пунктуацию, как периоды, восклицательные знаки и кавычки.

Итак, прежде чем мы пойдем дальше, давайте разберемся с Dashes и другими пунктуациями.

## пунктуация

Чтобы определить слова в тексте, нам нужно решить два вопроса:

* Когда в линии появляется черта, мы должны заменить ее пространством - затем, когда мы используем `split`, слова будут разделены.

* После разделения слов мы можем использовать `strip 'для удаления пунктуации.

Чтобы решить первую проблему, мы можем использовать следующую функцию, которая принимает строку, заменяет тире на пробелы, разбивает строку и возвращает полученный список.

In [11]:
def split_line(line):
    return line.replace('—', ' ').split()

Обратите внимание, что `split_line 'только заменяет панели, а не дефисы.
Вот пример.

In [12]:
split_line('coolness—frightened')

Теперь, чтобы удалить пунктуацию с самого начала и конца каждого слова, мы можем использовать «полосу», но нам нужен список символов, которые считаются пунктуацией.

Персонажи в струнах Python находятся в Unicode, который является международным стандартом, используемым для представления букв почти в каждом алфавите, числах, символах, знаках пунктуации и многом другом.
Модуль `UnicodeData` обеспечивает функцию« категория », которую мы можем использовать, чтобы сказать, какие символы являются пунктуацией.
Учитывая письмо, он возвращает строку с информацией о категории, в которой находится письмо.

In [13]:
import unicodedata

unicodedata.category('A')

Строка категории `'a'`` `lu'' -` 'l'`означает, что это буква, а« u »означает, что это - верх.

Строка категории `'.'` '' Po'` - «p'» означает, что это пунктуация, а «O'» означает, что его подкатегория - «Другое».

In [14]:
unicodedata.category('.')

Мы можем найти знаки препинания в книге, проверив персонажей с категориями, которые начинаются с «p'».
Следующая петля хранит уникальные знаки препинания в словаре.

In [15]:
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

Чтобы составить список знаков препинания, мы можем присоединиться к ключам словаря в строку.

In [16]:
punctuation = ''.join(punc_marks)
print(punctuation)

Теперь, когда мы знаем, какие персонажи в книге являются пунктуацией, мы можем написать функцию, которая берет слово, линит пунктуацию с самого начала и конец и преобразует ее в нижний регистр.

In [17]:
def clean_word(word):
    return word.strip(punctuation).lower()

Вот пример.

In [18]:
clean_word('“Behold!”')

Поскольку `strip 'удаляет символы с самого начала и конец, он оставляет дефисацию в покое.

In [19]:
clean_word('pocket-handkerchief')

Теперь вот цикл, который использует `split_line` и` clean_word`, чтобы определить уникальные слова в книге.

In [20]:
unique_words2 = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        unique_words2[word] = 1

len(unique_words2)

С этим более строгим определением того, что такое слово, существует около 4000 уникальных слов.
И мы можем подтвердить, что список самых длинных слов был очищен.

In [21]:
sorted(unique_words2, key=len)[-5:]

Теперь давайте посмотрим, сколько раз используется каждое слово.

## частоты слов

Следующий цикл вычисляет частоту каждого уникального слова.

In [22]:
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

В первый раз, когда мы видим слово, мы инициализируем его частоту до `1`. Если мы снова увидим то же слово, мы увеличиваем его частоту.

Чтобы увидеть, какие слова появляются чаще всего, мы можем использовать «элементы», чтобы получить пары клавиш значений от `word_counter` и отсортировать их по вторым элементам пары, которая является частотой.
Сначала мы определим функцию, которая выбирает второй элемент.

In [23]:
def second_element(t):
    return t[1]

Теперь мы можем использовать `sorted` с двумя аргументами ключевого слова:

* `key = second_element` означает, что элементы будут отсортированы в соответствии с частотами слов.

* `reverse = true` означает, что элементы будут отсортированы в обратном порядке, сначала с наиболее частыми словами.

In [24]:
items = sorted(word_counter.items(), key=second_element, reverse=True)

Вот пять наиболее частых слов.

In [25]:
for word, freq in items[:5]:
    print(freq, word, sep='\t')

В следующем разделе мы инкапсулируем этот цикл в функцию.
И мы будем использовать его, чтобы продемонстрировать новую функцию - дополнительные параметры.

## необязательные параметры

Мы использовали встроенные функции, которые принимают дополнительные параметры.
Например, «Round» принимает дополнительные параметры, называемые «ndigits», которые указывают на то, сколько десятичных мест сохранить.

In [26]:
round(3.141592653589793, ndigits=3)

Но это не просто встроенные функции-мы также можем написать функции с дополнительными параметрами.
Например, следующая функция принимает два параметра: `word_counter` и` num`.

In [27]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

Второй параметр выглядит как оператор назначения, но это не так - это необязательный параметр.

Если вы называете эту функцию одним аргументом, `num` получает значение ** по умолчанию **, которое равно` 5`.

In [28]:
print_most_common(word_counter)

Если вы называете эту функцию с двумя аргументами, второй аргумент назначается «num» вместо значения по умолчанию.

In [29]:
print_most_common(word_counter, 3)

В этом случае мы бы сказали, что необязательный аргумент ** переопределяет ** значение по умолчанию.

Если функция имеет как требуемые, так и дополнительные параметры, все необходимые параметры должны быть первыми, а затем дополнительные.

In [30]:
%%expect SyntaxError

def bad_function(n=5, word_counter):
    return None

## Словарь вычитание

Предположим, мы хотим проверить книгу, то есть найти список слов, которые могут быть написаны с ошибкой.
Один из способов сделать это - найти слова в книге, которые не появляются в списке действительных слов.
В предыдущих главах мы использовали список слов, которые считаются действительными в таких словах, как Scrabble.
Теперь мы будем использовать этот список для проверки орфографии Роберта Луи Стивенсона.

Мы можем думать об этой проблеме как о вычитании установки, то есть мы хотим найти все слова из одного набора (слова в книге), которые не находятся в другом (слова в списке).

Следующая ячейка загружает список слов.

In [31]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Как мы делали раньше, мы можем прочитать содержимое `words.txt` и разделить его на список строк.

In [32]:
word_list = open('words.txt').read().split()

Затем мы сохраним слова в качестве ключей в словаре, чтобы мы могли использовать оператора «в», чтобы быстро проверить, является ли слово достоверным.

In [33]:
valid_words = {}
for word in word_list:
    valid_words[word] = 1

Теперь, чтобы идентифицировать слова, которые появляются в книге, но не в списке слов, мы будем использовать «вычтенный», который принимает два словаря в качестве параметров и возвращает новый словарь, который содержит все ключи от одного, которых нет в другом.

In [34]:
def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res

Вот как мы его используем.

In [35]:
diff = subtract(word_counter, valid_words)

Чтобы получить образец слов, которые могут быть сформулированы, мы можем распечатать наиболее распространенные слова в «diff».

In [36]:
print_most_common(diff)

Наиболее распространенными словами «неправильно» являются в основном имена и несколько однобуквенных слов (мистер Уттерсон-друг и адвокат доктора Джекилла).

Если мы выбираем слова, которые появляются только один раз, они с большей вероятностью будут фактическими ошибками.
Мы можем сделать это, пробежая элементы и составив список слов с частотой `1`.

In [37]:
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

Вот последние несколько элементов списка.

In [38]:
singletons[-5:]

Большинство из них являются действительными словами, которых нет в списке слов.
Но «охрана», по -видимому, является неправильным написанием «повторного положения», поэтому, по крайней мере, мы нашли одну законную ошибку.

## случайные числа

В качестве шага к генерации текста Маркова, затем мы выберем случайную последовательность слов из `word_counter`.
Но сначала давайте поговорим о случайности.

Учитывая те же входные данные, большинство компьютерных программ ** детерминированные **, что означает, что они генерируют одни и те же выходы каждый раз.
Обычно детерминизм - это хорошая вещь, поскольку мы ожидаем, что тот же расчет будет даст тот же результат.
Однако для некоторых приложений мы хотим, чтобы компьютер был непредсказуемым.
Игры - один из примеров, но есть еще больше.

Сделать программу по -настоящему нетерминированной оказывается трудной, но есть способы ее подделать.
Одним из них является использование алгоритмов, которые генерируют ** псевдорандома ** числа.
Числа псевдордомов не являются по -настоящему случайными, потому что они генерируются детерминированными вычислениями, но, просто глядя на числа, почти невозможно отличить их от случайных.

Модуль «случайный» предоставляет функции, которые генерируют числа псевдордомов, которые я просто назову «случайным» с этого момента.
Мы можем импортировать его так.

In [39]:
import random

In [40]:
# this cell initializes the random number generator so it 
# generates the same sequence each time the notebook runs.

random.seed(4)

Модуль «случайный» предоставляет функцию, называемую «выбор», которая выбирает элемент из списка случайным образом, причем каждый элемент имеет одинаковую вероятность выбора.

In [41]:
t = [1, 2, 3]
random.choice(t)

Если вы снова вызовите функцию, вы можете снова получить один и тот же элемент или другой.

In [42]:
random.choice(t)

В долгосрочной перспективе мы ожидаем получить каждый элемент примерно одинаково раз.

Если вы используете «выбор» с словарем, вы получите `keeError '.

In [43]:
%%expect KeyError

random.choice(word_counter)

Чтобы выбрать случайный ключ, вы должны поместить клавиши в список, а затем позвонить «выбор».

In [44]:
words = list(word_counter)
random.choice(words)

Если мы генерируем случайную последовательность слов, это не имеет большого смысла.

In [45]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

Часть проблемы в том, что мы не учитываем, что некоторые слова чаще, чем другие.
Результаты будут лучше, если мы выберем слова с различными «весами», чтобы некоторые из них выбирались чаще, чем другие.

Если мы используем значения из `word_counter` в качестве веса, каждое слово выбирается с вероятностью, которая зависит от его частоты.

In [46]:
weights = word_counter.values()

Модуль `случайный 'предоставляет другую функцию под названием« Выбор », которая принимает веса в качестве необязательного аргумента.

In [47]:
random.choices(words, weights=weights)

И требуется еще один необязательный аргумент, `k`, который указывает количество слов для выбора.

In [48]:
random_words = random.choices(words, weights=weights, k=6)
random_words

Результатом является список строк, которые мы можем присоединиться к чем -то, что больше похоже на предложение.

In [49]:
' '.join(random_words)

Если вы выбираете слова из книги случайным образом, вы понимаете словарный запас, но серия случайных слов редко имеет смысл, потому что нет никакой связи между последовательными словами.
Например, в реальном предложении вы ожидаете, что статья, такая как «The», за которой следует прилагательное или существительное, и, вероятно, не глагол или наречие.
Таким образом, следующий шаг - взглянуть на эти отношения между словами.

## Bigrams

Вместо того, чтобы смотреть на одно слово за раз, теперь мы рассмотрим последовательности из двух слов, которые называются ** Bigrams **.
Последовательность из трех слов называется ** триграмма **, а последовательность с некоторым неустановленным количеством слов называется ** n-грамм **.

Давайте напишем программу, которая находит все биграмы в книге, и количество раз каждый раз.
Чтобы сохранить результаты, мы будем использовать словарь, где

* Ключи - это корты струн, которые представляют биграмс, и 

* Значения - это целые числа, которые представляют частоты.

Назовем это `bigram_counter`.

In [50]:
bigram_counter = {}

Следующая функция принимает список из двух строк в качестве параметра.
Сначала он делает кортеж из двух струн, которые можно использовать в качестве ключа в словаре.
Затем он добавляет ключ к `bigram_counter`, если он не существует, или увеличивает частоту, если она это делает.

In [51]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

Когда мы проходим через книгу, мы должны отслеживать каждую пару последовательных слов.
Поэтому, если мы увидим последовательность «Человек не один», мы бы добавили биграмс «человек», «не», «не по -настоящему» и так далее.

Чтобы отслеживать эти Bigrams, мы будем использовать список под названием «Window», потому что это похоже на окно, которое скользит по страницам книги, показывая только два слова за раз.
Первоначально `window 'пусто.

In [52]:
window = []

Мы будем использовать следующую функцию для обработки слов по одному.

In [53]:
def process_word(word):
    window.append(word)
    
    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

В первый раз, когда эта функция называется, она добавляет данное слово в `window '.
Поскольку в окне есть только одно слово, у нас еще нет биграма, поэтому функция заканчивается.

Во второй раз это называется - и каждый раз после этого - он добавляет второе слово в «окно».
Поскольку в окне есть два слова, он называет `count_bigram`, чтобы отслеживать, сколько раз появляется каждый биграм.
Затем он использует `pop`, чтобы удалить первое слово из окна.

Следующая программа проходит через слова в книге и обрабатывает их по одному.

In [54]:
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

Результатом является словарь, который отображает от каждого биграма до количества раз, когда он появляется.
Мы можем использовать `print_sty_common`, чтобы увидеть наиболее распространенные биграмы.

In [55]:
print_most_common(bigram_counter)

Глядя на эти результаты, мы можем понять, какие пары слов, скорее всего, появятся вместе.
Мы также можем использовать результаты для генерации случайного текста, как это.

In [56]:
random.seed(0)

In [57]:
bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

`Bigrams ' - это список биграмсов, которые появляются в книгах.
`Weews` - это список их частот, поэтому` random_bigrams` - это выборка, в которой вероятность выбранности биграм пропорциональна его частоте. 

Вот результаты.

In [58]:
for pair in random_bigrams:
    print(' '.join(pair), end=' ')

Этот способ создания текста лучше, чем выбирать случайные слова, но все еще не имеет большого смысла.

## Анализ Маркова

Мы можем добиться большего успеха с анализом текста цепи Маркова, который вычисляет для каждого слова в тексте, список слов, которые следуют дальше.
Например, мы проанализируем эти тексты из песни Monty Python *Eric, половина пчелы *:

In [59]:
song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

Чтобы сохранить результаты, мы будем использовать словарь, который отображает из каждого слова в список слов, которые следуют за ним.

In [60]:
successor_map = {}

В качестве примера, давайте начнем с первых двух слов песни.

In [61]:
first = 'half'
second = 'a'

Если первое слово не в `customor_map`, мы должны добавить новый элемент, который отображает из первого слова в список, содержащий второе слово.

In [62]:
successor_map[first] = [second]
successor_map

Если первое слово уже в словаре, мы можем посмотреть его, чтобы получить список преемников, которых мы видели до сих пор, и добавить новый.

In [63]:
first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

Следующая функция инкапсулирует эти шаги.

In [64]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Если тот же биграм появляется больше, чем один раз, второе слово добавляется в список более одного раза.
Таким образом, `custeror_map` отслеживает, сколько раз появляется каждый преемник.

Как и в предыдущем разделе, мы будем использовать список под названием «Window» для хранения пар последовательных слов.
И мы будем использовать следующую функцию для обработки слов по одному.

In [65]:
def process_word_bigram(word):
    window.append(word)
    
    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

Вот как мы используем его для обработки слов в песне.

In [66]:
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

И вот результаты.

In [67]:
successor_map

Слово «полу» может сопровождаться `'a' ',`' not '' или `''.
Слово «a» может сопровождаться `'bee'' 'или`' vis '.
Большинство других слов появляются только один раз, поэтому за ними следует только одно слово.

Теперь давайте проанализируем книгу.

In [68]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

Мы можем искать любое слово и найти слова, которые могут следовать ему.

In [69]:
# I used this cell to find a predecessor with a good number of possible successors
# and at least one repeated word.

def has_duplicates(t):
    return len(set(t)) < len(t)

for key, value in successor_map.items():
    if len(value) == 7 and has_duplicates(value):
        print(key, value)

In [70]:
successor_map['going']

В этом списке преемников обратите внимание, что слово «до» появляется три раза - другие преемники появляются только один раз.

## генерирование текста

Мы можем использовать результаты из предыдущего раздела для создания нового текста с теми же отношениями между последовательными словами, что и в оригинале.
Вот как это работает:

* Начиная с любого слова, которое появляется в тексте, мы смотрим на его возможных преемников и выбираем его случайным образом.

* Затем, используя выбранное слово, мы смотрим его возможных преемников и выбираем его случайным образом.

Мы можем повторить этот процесс, чтобы генерировать столько слов, сколько хотим.
В качестве примера, давайте начнем со слова «хотя».
Вот слова, которые могут следовать за ним.

In [71]:
word = 'although'
successors = successor_map[word]
successors

In [72]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(2)

Мы можем использовать «выбор», чтобы выбрать из списка с одинаковой вероятностью.

In [73]:
word = random.choice(successors)
word

Если одно и то же слово появляется более одного раза в списке, оно будет выбран.

Повторяя эти шаги, мы можем использовать следующий цикл для создания более длинной серии.

In [74]:
for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

Результат звучит больше как реальное предложение, но это все еще не имеет большого смысла.

Мы можем лучше использовать более одного слова в качестве ключа в `curstor_map`.
Например, мы можем сделать словарь, который карты из каждой биграм - или триграммы - в список слов, которые приходят дальше.
В качестве упражнения у вас будет возможность реализовать этот анализ и посмотреть, как выглядят результаты.

## отладка

На данный момент мы пишем более существенные программы, и вы можете обнаружить, что вы тратите больше времени на отладку.
Если вы застряли на сложной ошибке, вот несколько вещей, которые нужно попробовать:

* Чтение: проверьте свой код, прочитайте его обратно и проверьте, что он говорит то, что вы хотели сказать.

* Запуск: Экспериментируйте, внося изменения и запустив разные версии. Часто, если вы отображаете правильные вещи в нужном месте в программе, проблема становится очевидной, но иногда вам приходится строить леса.

* Размышления: понадобится время, чтобы подумать! Что это за ошибка: синтаксис, время выполнения,
    или семантический? Какая информация вы можете получить от сообщений об ошибках,
    Или из вывода программы? Какая ошибка может вызвать
    Проблема, которую вы видите? Что вы изменили в последний раз, до
    Появилась проблема?

* RubberDucking: Если вы объясните проблему кому -то другому, вы иногда найдете
    Ответьте, прежде чем закончить задание вопроса. Часто вам не нужно
    другой человек; Вы могли бы просто поговорить с резиновой уткой. И это
    Происхождение известной стратегии под названием ** резиновая утка
    отладка **. Я не придумываю это - вижу
    <https://en.wikipedia.org/wiki/rubber_duck_debugging>.

* Отступление: в какой -то момент лучше
    Изменения - пока вы не доберетесь до программы, которая работает. Тогда вы можете начать восстановление.
    
* Отдых: если вы дадите своему мозгу разрыв, иногда это найдет проблему для вас.

Начальные программисты иногда застряли на одном из этих действий и забывают других. Каждое действие поставляется с собственным режимом отказа.

Например, чтение вашего кода работает, если проблема является типографской ошибкой, но не в том случае, если проблема является концептуальным недопониманием.
Если вы не понимаете, что делает ваша программа, вы можете прочитать ее 100 раз и никогда не увидеть ошибку, потому что ошибка у вас в голове.

Запуск экспериментов может работать, особенно если вы запускаете маленькие, простые тесты.
Но если вы запускаете эксперименты, не задумываясь и не читая свой код, может потребоваться много времени, чтобы выяснить, что происходит.

Вы должны понадобиться, чтобы подумать. Отладка похожа на экспериментальную науку. У вас должна быть хотя бы одна гипотеза о том, в чем проблема. Если есть две или более возможностей, попробуйте придумать тест, который устранит один из них.

Но даже лучшие методы отладки потерпят неудачу, если их слишком много
Ошибки, или если код, который вы пытаетесь исправить, слишком большой и сложный.
Иногда лучший вариант - отступить, упрощая программу до тех пор, пока
Вы возвращаетесь к чему -то, что работает.

Начальные программисты часто неохотно отступают, потому что они не могут
Встаньте, чтобы удалить строку кода (даже если это не так). Если это заставит вас
Почувствуйте себя лучше, скопируйте свою программу в другой файл, прежде чем начать
Разбрасывая это. Тогда вы можете скопировать части обратно по одному.

Поиск жесткой ошибки требует чтения, бега, размышлений, отступления и иногда отдыха.
Если вы застряли на одном из этих действий, попробуйте другие.

## Глоссарий

** Значение по умолчанию: **
Значение, присвоенное параметру, если аргумент не предоставлен.

** Переопределение: **
 Чтобы заменить значение по умолчанию на аргумент.

** Детерминированный: **
 Детерминистская программа делает одно и то же каждый раз, когда она работает, учитывая одинаковые входы.

** псевдорандома: **
 Последовательность псевдорендомов чисел, по -видимому, является случайной, но генерируется детерминистской программой.

** Bigram: **
Последовательность двух элементов, часто слова.

** Триграмма: **
Последовательность из трех элементов.

** n-gram: **
Последовательность неустановленного числа элементов.

** Отладка резиновой утки: **
Способ отладки, объяснив проблему вслух неодушевленным объектам.

## Упражнения

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Спросите виртуального помощника

В `add_bigram` оператор« if »создает новый список или добавляет элемент к существующему списку, в зависимости от того, находится ли ключ в словаре.

In [75]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Словары предоставляют метод под названием «SetDefault», который мы можем использовать, чтобы сделать то же самое более кратко.
Спросите виртуального помощника, как это работает, или скопируйте `add_bigram` в виртуального помощника и спросите:« Можете ли вы переписать это, используя `setdefault`?»

В этой главе мы реализовали анализ текста и генерации цепи Маркова.
Если вам любопытно, вы можете попросить виртуального помощника для получения дополнительной информации по теме.
Одна из вещей, которые вы можете узнать, заключается в том, что виртуальные помощники используют алгоритмы, которые во многих отношениях похожи, но также различны важными способами.
Спросите VA: «Каковы различия между большими языковыми моделями, такими как анализ текста GPT и марковского текста?»

### Упражнение

Напишите функцию, которая подсчитывает количество раз, когда появляется каждая триграмма (последовательность трех слов). 
Если вы проверяете свою функцию с помощью текста _dr. Jekyll и Mr. Hyde_, вы должны обнаружить, что наиболее распространенной триграммой является «сказал адвокат».

Подсказка: напишите функцию с именем `count_trigram`, которая похожа на` count_bigram`. Затем напишите функцию с именем `process_word_trigram`, которая похожа на` process_word_bigram`.

In [76]:
# Solution goes here

In [77]:
# Solution goes here

Вы можете использовать следующий цикл, чтобы прочитать книгу и обработать слова.

In [78]:
trigram_counter = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Затем используйте `print_sty_common`, чтобы найти наиболее распространенные триграммы в книге.

In [79]:
print_most_common(trigram_counter)

### Упражнение

Теперь давайте внедрим анализ текста цепи Маркова с отображением от каждого биграма в список возможных преемников.

Начиная с `add_bigram`, напишите функцию под названием` add_trigram`, которая принимает список из трех слов и добавляет или обновляет элемент в `custcortor_map`, используя первые два слова в качестве ключа и третье слово в качестве возможного преемника.

In [80]:
# Solution goes here

Вот версия `process_word_trigram`, которая вызывает` add_trigram`.

In [81]:
def process_word_trigram(word):
    window.append(word)
    
    if len(window) == 3:
        add_trigram(window)
        window.pop(0)

Вы можете использовать следующий цикл, чтобы проверить свою функцию с помощью текста «Эрик, половина пчелы».

In [82]:
successor_map = {}
window = []

for string in song.split():
    word = string.strip(punctuation).lower()
    process_word_trigram(word)

Если ваша функция работает так, как предназначена, предшественник `(« половина », 'a')` должен составить карту в список с одним элементом `'bee''.
На самом деле, как это происходит, каждый биграм в этой песне появляется только один раз, поэтому все значения в `custcoror_map` имеют единый элемент.

In [83]:
successor_map

Вы можете использовать следующий цикл, чтобы проверить свою функцию со словами из книги.

In [84]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

В следующем упражнении вы используете результаты для генерации нового случайного текста.

### Упражнение

Для этого упражнения мы предположим, что `custoror_map` - это словарь, который отображает из каждого биграма в список слов, которые следуют за ним.

In [85]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(3)

Чтобы сгенерировать случайный текст, мы начнем с выбора случайного ключа из `customor_map`.

In [86]:
successors = list(successor_map)
bigram = random.choice(successors)
bigram

Теперь напишите цикл, который генерирует еще 50 слов после этих шагов:

1. В `custeror_map` найдите список слов, которые могут следовать за` bigram '.

2. Выберите один из них случайным образом и распечатайте его.

3. Для следующей итерации сделайте новый биграм, который содержит второе слово от `Bigram` и выбранного преемника.

Например, если мы начнем с Bigram `('сомневались', 'if')` и выберем `'from' 'в качестве преемника, следующий биграм - (' if ',' from ')`.

In [89]:
# Solution goes here

Если все работает, вы должны обнаружить, что сгенерированный текст узнаваемо по -настоящему похож на оригинал, а некоторые фразы имеют смысл, но текст может бродить от одной темы к другой.

В качестве бонусного упражнения измените свое решение до двух последних упражнений, чтобы использовать триграммы в качестве ключей в `custormor_map`, и посмотрите, какое влияние он оказывает на результаты.

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Лицензия на код: [MIT License](https://mit-license.org/)

Текстовая лицензия: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)